# Fungi Interpolation Lab — Pangea-Earth

**The between IS the product. The in-between states are the game.**

1. Mycelium growth between two sprites — the network IS the interpolation
2. Morphological interpolation — blend silhouettes at any t from 0.0 to 1.0
3. Depth map interpolation — MiDaS heightmaps blend to create 3D in-between forms
4. Color palette interpolation — melanin ratio slider between any two characters
5. Turing texture interpolation — spots morph to stripes morph to labyrinth

The fungi don't jump between nodes. They grow through the space between them.
The contract forms at the connection point. The interpolation IS the mycelium.

Guinea Pig Trench LLC

In [ ]:
#@title 1. Setup — Mount Drive, load sprites
import os, time, json, io
import numpy as np
import cv2
from pathlib import Path
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

!pip install -q rembg onnxruntime-gpu Pillow==11.2.1

from PIL import Image

DRIVE_BASE = Path('/content/drive/MyDrive/Guinea Pig Trench')
SOURCE_DIR = DRIVE_BASE / 'sprites' / 'source'
OUTPUT_DIR = DRIVE_BASE / 'sprites' / 'interpolated'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SESSION_START = time.time()
SESSION_LIMIT = 80 * 60
def time_left(): return max(0, SESSION_LIMIT - (time.time() - SESSION_START))

# Load all source sprites
sources = {}
for f in sorted(SOURCE_DIR.glob('*.png')):
    img = Image.open(f).convert('RGBA')
    sources[f.stem] = img
    print(f'  {f.stem}: {img.size[0]}x{img.size[1]}')

print(f'\n{len(sources)} sprites loaded')
print(f'Session budget: {SESSION_LIMIT//60} min')

In [ ]:
#@title 2. Mycelium Growth Between Two Sprites

def grow_between(img_a, img_b, steps=200, n_tips=12):
    """Grow mycelium from sprite A toward sprite B.
    The network that forms IS the interpolation.
    Each frame captures the mycelium at a different growth stage."""
    
    # Resize both to same dimensions
    size = (256, 256)
    a = np.array(img_a.resize(size, Image.LANCZOS).convert('RGBA'), dtype=np.float32) / 255
    b = np.array(img_b.resize(size, Image.LANCZOS).convert('RGBA'), dtype=np.float32) / 255
    
    # Canvas — starts as sprite A
    canvas = a.copy()
    
    # Signal source = pixels where B differs most from A
    diff = np.sqrt(np.sum((b[:,:,:3] - a[:,:,:3])**2, axis=2))
    signal = diff / (diff.max() + 1e-8)
    
    # Mycelium tips start at random positions on A's visible region
    a_alpha = a[:,:,3]
    a_visible = np.where(a_alpha > 0.1)
    if len(a_visible[0]) == 0:
        return []
    
    tips = []
    for _ in range(n_tips):
        idx = np.random.randint(len(a_visible[0]))
        tips.append({
            'y': float(a_visible[0][idx]),
            'x': float(a_visible[1][idx]),
            'angle': np.random.uniform(0, 2*np.pi),
            'energy': 1.0,
        })
    
    frames = []
    
    for step in range(steps):
        t = step / max(steps - 1, 1)  # 0 to 1
        
        # Global blend — canvas drifts from A toward B
        canvas = a * (1 - t) + b * t
        
        # Mycelium tips grow toward high-signal (high-difference) regions
        new_tips = []
        for tip in tips:
            if tip['energy'] <= 0:
                continue
            
            # Sense signal
            iy, ix = int(np.clip(tip['y'], 0, size[1]-1)), int(np.clip(tip['x'], 0, size[0]-1))
            local_signal = signal[iy, ix]
            
            # Steer toward gradient
            if iy > 0 and iy < size[1]-1 and ix > 0 and ix < size[0]-1:
                gx = signal[iy, min(ix+1, size[0]-1)] - signal[iy, max(ix-1, 0)]
                gy = signal[min(iy+1, size[1]-1), ix] - signal[max(iy-1, 0), ix]
                target_angle = np.arctan2(gy, gx)
                angle_diff = target_angle - tip['angle']
                angle_diff = (angle_diff + np.pi) % (2*np.pi) - np.pi
                tip['angle'] += angle_diff * 0.4
            
            tip['angle'] += np.random.normal(0, 0.15)
            
            # Move
            speed = 1.5
            tip['x'] += np.cos(tip['angle']) * speed
            tip['y'] += np.sin(tip['angle']) * speed
            tip['energy'] -= 0.003
            
            # At the tip position, accelerate the blend toward B
            iy2 = int(np.clip(tip['y'], 0, size[1]-1))
            ix2 = int(np.clip(tip['x'], 0, size[0]-1))
            r = 3
            y1, y2 = max(0, iy2-r), min(size[1], iy2+r+1)
            x1, x2 = max(0, ix2-r), min(size[0], ix2+r+1)
            blend_boost = min(1.0, t + 0.3)
            canvas[y1:y2, x1:x2] = canvas[y1:y2, x1:x2] * (1-0.1) + b[y1:y2, x1:x2] * 0.1
            
            if 0 <= tip['x'] < size[0] and 0 <= tip['y'] < size[1]:
                new_tips.append(tip)
                
                # Branch
                if step % 30 == 0 and tip['energy'] > 0.3 and local_signal > 0.2:
                    new_tips.append({
                        'x': tip['x'], 'y': tip['y'],
                        'angle': tip['angle'] + np.random.choice([-1,1]) * np.random.uniform(0.3, 0.7),
                        'energy': tip['energy'] * 0.7
                    })
        
        tips = new_tips
        
        # Save keyframes
        if step % (steps // 8) == 0 or step == steps - 1:
            frame = (np.clip(canvas, 0, 1) * 255).astype(np.uint8)
            frames.append(Image.fromarray(frame))
    
    return frames


# Grow between first two sprites
sprite_names = list(sources.keys())
if len(sprite_names) >= 2:
    name_a, name_b = sprite_names[0], sprite_names[1]
    print(f'=== Mycelium Growth: {name_a} → {name_b} ===')
    frames = grow_between(sources[name_a], sources[name_b], steps=200)
    
    if frames:
        fig, axes = plt.subplots(1, len(frames), figsize=(3*len(frames), 3))
        for i, frame in enumerate(frames):
            axes[i].imshow(np.array(frame))
            axes[i].set_title(f't={i/(len(frames)-1):.2f}', fontsize=8)
            axes[i].axis('off')
            frame.save(str(OUTPUT_DIR / f'{name_a}_to_{name_b}_t{i:02d}.png'))
        plt.suptitle(f'Mycelium Growth: {name_a} → {name_b}', fontsize=12)
        plt.tight_layout()
        plt.savefig(str(OUTPUT_DIR / f'mycelium_{name_a}_to_{name_b}.png'), dpi=150)
        plt.show()
        print(f'{len(frames)} keyframes saved to {OUTPUT_DIR}')

In [ ]:
#@title 3. Morphological Interpolation — silhouette blending

def morph_silhouettes(img_a, img_b, n_steps=9):
    """Distance transform interpolation between two silhouettes.
    The in-between shapes are forms that don't exist in either sprite.
    The between IS the product."""
    
    size = (256, 256)
    a = np.array(img_a.resize(size).convert('L'), dtype=np.float32)
    b = np.array(img_b.resize(size).convert('L'), dtype=np.float32)
    
    # Threshold to binary
    a_bin = (a > 30).astype(np.uint8)
    b_bin = (b > 30).astype(np.uint8)
    
    # Distance transforms — how far each pixel is from the edge
    a_dist = cv2.distanceTransform(a_bin, cv2.DIST_L2, 5).astype(np.float32)
    b_dist = cv2.distanceTransform(b_bin, cv2.DIST_L2, 5).astype(np.float32)
    a_dist_inv = cv2.distanceTransform(1-a_bin, cv2.DIST_L2, 5).astype(np.float32)
    b_dist_inv = cv2.distanceTransform(1-b_bin, cv2.DIST_L2, 5).astype(np.float32)
    
    # Signed distance fields
    a_sdf = a_dist - a_dist_inv
    b_sdf = b_dist - b_dist_inv
    
    frames = []
    for i in range(n_steps):
        t = i / (n_steps - 1)
        
        # Interpolate SDFs
        sdf_blend = a_sdf * (1 - t) + b_sdf * t
        
        # Threshold back to binary
        morph = (sdf_blend > 0).astype(np.uint8) * 255
        
        # Color interpolation from both source sprites
        a_rgba = np.array(img_a.resize(size).convert('RGBA'), dtype=np.float32) / 255
        b_rgba = np.array(img_b.resize(size).convert('RGBA'), dtype=np.float32) / 255
        color_blend = a_rgba * (1 - t) + b_rgba * t
        
        # Apply morphed silhouette as alpha
        color_blend[:,:,3] = morph.astype(np.float32) / 255
        
        frame = (np.clip(color_blend, 0, 1) * 255).astype(np.uint8)
        frames.append(Image.fromarray(frame))
    
    return frames


# Generate morphs between multiple pairs
print('=== Morphological Interpolation ===')
print('SDF blending — shapes that exist between characters')
print()

pairs = []
for i in range(min(len(sprite_names)-1, 3)):
    pairs.append((sprite_names[i], sprite_names[i+1]))

for name_a, name_b in pairs:
    if time_left() < 120:
        break
    print(f'  {name_a} ↔ {name_b}:')
    frames = morph_silhouettes(sources[name_a], sources[name_b], n_steps=9)
    
    fig, axes = plt.subplots(1, len(frames), figsize=(2*len(frames), 2))
    for j, frame in enumerate(frames):
        axes[j].imshow(np.array(frame))
        t = j / (len(frames)-1)
        axes[j].set_title(f'{t:.1f}', fontsize=7)
        axes[j].axis('off')
        frame.save(str(OUTPUT_DIR / f'morph_{name_a}_{name_b}_t{j:02d}.png'))
    plt.suptitle(f'{name_a} → {name_b} (SDF morph)', fontsize=10)
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / f'morph_strip_{name_a}_{name_b}.png'), dpi=150)
    plt.show()
    print(f'  {len(frames)} frames saved')

In [ ]:
#@title 4. Depth Interpolation — MiDaS heightmap blending (GPU)
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

midas = torch.hub.load('intel-isl/MiDaS', 'DPT_Large')
midas.to(device).eval()
midas_transforms = torch.hub.load('intel-isl/MiDaS', 'transforms')
transform = midas_transforms.dpt_transform


def get_depth(img):
    arr = np.array(img.convert('RGB'))
    input_batch = transform(arr).to(device)
    with torch.no_grad():
        pred = midas(input_batch)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1), size=arr.shape[:2], mode='bicubic', align_corners=False
        ).squeeze()
    depth = pred.cpu().numpy()
    depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)
    return depth


def interpolate_depths(img_a, img_b, n_steps=7):
    """Generate depth maps for both sprites, then interpolate.
    The in-between depth maps represent 3D forms that don't exist."""
    
    size = (256, 256)
    a_resized = img_a.resize(size, Image.LANCZOS)
    b_resized = img_b.resize(size, Image.LANCZOS)
    
    depth_a = get_depth(a_resized)
    depth_b = get_depth(b_resized)
    
    frames = []
    for i in range(n_steps):
        t = i / (n_steps - 1)
        depth_blend = depth_a * (1 - t) + depth_b * t
        frames.append(depth_blend)
    
    return frames, depth_a, depth_b


print('=== Depth Map Interpolation (MiDaS on GPU) ===')
print('3D forms that exist between two characters')
print()

if len(sprite_names) >= 2:
    name_a, name_b = sprite_names[0], sprite_names[1]
    print(f'  {name_a} ↔ {name_b}:')
    depth_frames, da, db = interpolate_depths(sources[name_a], sources[name_b], n_steps=7)
    
    fig, axes = plt.subplots(1, len(depth_frames), figsize=(2.5*len(depth_frames), 2.5))
    for j, df in enumerate(depth_frames):
        axes[j].imshow(df, cmap='magma')
        t = j / (len(depth_frames)-1)
        axes[j].set_title(f't={t:.1f}', fontsize=8)
        axes[j].axis('off')
        depth_img = (df * 255).astype(np.uint8)
        Image.fromarray(depth_img).save(str(OUTPUT_DIR / f'depth_{name_a}_{name_b}_t{j:02d}.png'))
    plt.suptitle(f'Depth Interpolation: {name_a} → {name_b}', fontsize=10)
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / f'depth_strip_{name_a}_{name_b}.png'), dpi=150)
    plt.show()
    print(f'  {len(depth_frames)} depth frames saved')

In [ ]:
#@title 5. Turing Pattern Interpolation — spots → stripes → labyrinth

def turing_interpolate(width=256, height=256, steps=3000, n_keyframes=7):
    """Interpolate Turing reaction-diffusion parameters.
    f and k slide continuously — the pattern morphs organically.
    Spots → stripes → labyrinth → mitosis → decay."""
    
    # Parameter path through (f, k) space
    params = [
        (0.035, 0.065),  # spots
        (0.042, 0.062),  # worms
        (0.055, 0.062),  # stripes
        (0.042, 0.059),  # labyrinth
        (0.060, 0.062),  # mitosis
        (0.035, 0.065),  # back to spots
    ]
    
    A = torch.ones(height, width, device=device)
    B = torch.zeros(height, width, device=device)
    cx, cy = width//2, height//2
    B[cy-15:cy+15, cx-15:cx+15] = 1.0
    B += torch.rand_like(B) * 0.05
    
    laplacian = torch.tensor([[0.05, 0.2, 0.05],
                              [0.2, -1.0, 0.2],
                              [0.05, 0.2, 0.05]], device=device).unsqueeze(0).unsqueeze(0)
    
    keyframes = []
    keyframe_interval = steps // n_keyframes
    
    for step in range(steps):
        # Interpolate parameters along the path
        path_t = (step / steps) * (len(params) - 1)
        idx = int(path_t)
        frac = path_t - idx
        idx = min(idx, len(params) - 2)
        
        f = params[idx][0] * (1-frac) + params[idx+1][0] * frac
        k = params[idx][1] * (1-frac) + params[idx+1][1] * frac
        
        A_pad = torch.nn.functional.pad(A.unsqueeze(0).unsqueeze(0), (1,1,1,1), mode='circular')
        B_pad = torch.nn.functional.pad(B.unsqueeze(0).unsqueeze(0), (1,1,1,1), mode='circular')
        
        lap_A = torch.nn.functional.conv2d(A_pad, laplacian).squeeze()
        lap_B = torch.nn.functional.conv2d(B_pad, laplacian).squeeze()
        
        reaction = A * B * B
        A = A + (1.0 * lap_A - reaction + f * (1 - A))
        B = B + (0.5 * lap_B + reaction - (k + f) * B)
        A = torch.clamp(A, 0, 1)
        B = torch.clamp(B, 0, 1)
        
        if step % keyframe_interval == 0:
            b_np = B.cpu().numpy()
            # Melanin coloring
            eu = np.stack([b_np * 0.07, b_np * 0.045, b_np * 0.025], axis=-1)
            ph = np.stack([(1-b_np) * 0.68, (1-b_np) * 0.35, (1-b_np) * 0.15], axis=-1)
            ratio = b_np[:,:,np.newaxis]
            colored = np.clip((eu * ratio + ph * (1-ratio)) * 3, 0, 1)
            keyframes.append((colored, f, k))
    
    return keyframes


print('=== Turing Pattern Interpolation (GPU) ===')
print('Parameters slide: spots → worms → stripes → labyrinth → mitosis → spots')
print()

keyframes = turing_interpolate(256, 256, steps=3000, n_keyframes=8)

fig, axes = plt.subplots(1, len(keyframes), figsize=(3*len(keyframes), 3))
for i, (frame, f, k) in enumerate(keyframes):
    axes[i].imshow(frame)
    axes[i].set_title(f'f={f:.3f}\nk={k:.3f}', fontsize=7)
    axes[i].axis('off')
    img = (np.clip(frame, 0, 1) * 255).astype(np.uint8)
    Image.fromarray(img).save(str(OUTPUT_DIR / f'turing_interp_{i:02d}.png'))

plt.suptitle('Turing Interpolation with Melanin Coloring', fontsize=12)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'turing_interpolation_strip.png'), dpi=150)
plt.show()
print(f'{len(keyframes)} keyframes saved')

In [ ]:
#@title 6. Session Summary

all_outputs = list(OUTPUT_DIR.glob('*.png'))

results = {
    'session_date': time.strftime('%Y-%m-%d %H:%M'),
    'mycelium_growth_frames': len([f for f in all_outputs if 'mycelium' in f.stem or '_to_' in f.stem]),
    'morph_frames': len([f for f in all_outputs if 'morph' in f.stem]),
    'depth_frames': len([f for f in all_outputs if 'depth' in f.stem]),
    'turing_frames': len([f for f in all_outputs if 'turing' in f.stem]),
    'total_outputs': len(all_outputs),
    'duration_min': round((time.time() - SESSION_START) / 60, 1),
}

(OUTPUT_DIR / 'interpolation_session.json').write_text(json.dumps(results, indent=2))

print('=== Fungi Interpolation Lab — Session Summary ===')
print(f'Duration: {results["duration_min"]} min')
print(f'Mycelium growth frames: {results["mycelium_growth_frames"]}')
print(f'Morph frames: {results["morph_frames"]}')
print(f'Depth interpolation frames: {results["depth_frames"]}')
print(f'Turing interpolation frames: {results["turing_frames"]}')
print(f'Total outputs: {results["total_outputs"]}')
print(f'Time remaining: {time_left()/60:.0f} min')
print()
print('The between IS the product.')
print('The fungi grow through the space between nodes.')
print('The interpolation IS the mycelium.')
print('Pangea-Earth. Guinea Pig Trench LLC')